In [0]:
import pandas as pd
import os
import kagglehub

# Download latest version
path = kagglehub.dataset_download("dansbecker/powerlifting-database")

print("Path to dataset files:", path)

dataset_path = "/home/spark-6c0e28bd-0f58-426e-9d18-ab/.cache/kagglehub/datasets/binayas/powerlifting-dataset/versions/12"

# List all files in the dataset (recursively)
for root, dirs, files in os.walk(dataset_path):
    for f in sorted(files):
        rel = os.path.relpath(os.path.join(root, f), dataset_path)
        size_mb = os.path.getsize(os.path.join(root, f)) / 1024 / 1024
        print(f"{rel}  ({size_mb:.2f} MB)")

100%|██████████| 8.85M/8.85M [00:00<00:00, 67.4MB/s]

Extracting files...


Path to dataset files: /home/spark-0a1f5045-cdd6-415c-8431-fa/.cache/kagglehub/datasets/dansbecker/powerlifting-database/versions/1


In [0]:
import os
import pandas as pd
from pyspark.sql.functions import current_timestamp

# Load CSV files from the downloaded dataset into a dictionary of Spark DataFrames
# Using pandas to read local files (Spark can't access driver filesystem on Serverless)
dfs = {}
for f in os.listdir(path):
    if f.endswith(".csv"):
        table_name = f.replace(".csv", "")
        pdf = pd.read_csv(os.path.join(path, f))
        dfs[table_name] = spark.createDataFrame(pdf)
        print(f"Loaded {table_name} ({dfs[table_name].count()} rows)")

# Create the powerlifting_bronze schema
spark.sql("CREATE SCHEMA IF NOT EXISTS powerlifting_bronze")

# Save every table into the powerlifting_bronze schema
# Add _ingested_at timestamp for load provenance and enable Delta CDF so future incremental runs can detect what changed (inserts/updates/deletes).
# Write mode is APPEND - each run adds the latest Kaggle snapshot as a new batch.
# Duplicates are expected (same rows re-loaded); silver layer deduplicates via MERGE.
for name, df in dfs.items():
    df.withColumn("_ingested_at", current_timestamp()) \
     .write.format("delta") \
     .option("mergeSchema", "true") \
     .mode("append") \
     .saveAsTable(f"powerlifting_bronze.{name}")
    print(f"Appended to powerlifting_bronze.{name} (with _ingested_at)")

# Enable Delta Change Data Feed (CDF) on all bronze tables via ALTER TABLE
for name in dfs.keys():
    spark.sql(f"ALTER TABLE powerlifting_bronze.{name} SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')")
    print(f"CDF enabled on powerlifting_bronze.{name}")

# Verify
display(spark.sql("SHOW TABLES IN powerlifting_bronze"))

# Quick check: show CDF + ingestion column on one table
display(spark.sql("DESCRIBE TABLE EXTENDED powerlifting_bronze.openpowerlifting").filter("col_name IN ('_ingested_at', 'delta.enableChangeDataFeed')"))

Loaded meets (8482 rows)
Loaded openpowerlifting (386414 rows)
Appended to powerlifting_bronze.meets (with _ingested_at)
Appended to powerlifting_bronze.openpowerlifting (with _ingested_at)
CDF enabled on powerlifting_bronze.meets
CDF enabled on powerlifting_bronze.openpowerlifting


database,tableName,isTemporary
powerlifting_bronze,meets,false
powerlifting_bronze,openpowerlifting,false


col_name,data_type,comment
_ingested_at,timestamp,null


In [0]:
%sql
SELECT * FROM workspace.powerlifting_bronze.meets;
SELECT * FROM workspace.powerlifting_bronze.openpowerlifting

MeetID,Name,Sex,Equipment,Age,Division,BodyweightKg,WeightClassKg,Squat4Kg,BestSquatKg,Bench4Kg,BestBenchKg,Deadlift4Kg,BestDeadliftKg,TotalKg,Place,Wilks,_ingested_at
5094,Dave Dobin,M,Single-ply,null,Amateur Teen 18-19,73.48,75,null,185.97,null,117.93,null,213.19,517.1,1,373.83,2026-10-18T18:42:31.514Z
5094,Michael DeMeo,M,Wraps,null,Amateur Teen 18-19,80.56,82.5,null,199.58,null,117.93,null,206.38,523.9,2,356.11,2026-10-18T18:42:31.514Z
5094,Vincent Apicella,M,Wraps,null,Amateur Teen 18-19,82.1,82.5,null,226.8,null,138.35,null,206.38,571.53,1,383.99,2026-10-18T18:42:31.514Z
5094,Tyler Citta,M,Single-ply,null,Amateur Teen 16-17,80.38,82.5,null,151.95,null,111.13,null,185.97,449.06,2,305.66,2026-10-18T18:42:31.514Z
5094,Billy Geldhauser,M,Single-ply,null,Amateur Teen 16-17,79.83,82.5,null,188.24,null,127.01,null,210.92,526.17,1,359.7,2026-10-18T18:42:31.514Z
5094,Marty Stallone,M,Wraps,null,Amateur Teen 18-19,96.8,100,null,161.03,null,142.88,null,226.8,530.7,2,327.38,2026-10-18T18:42:31.514Z
5094,Shin Yongchul,M,Single-ply,null,Amateur Teen 18-19,97.7,100,null,204.12,null,113.4,null,229.06,546.58,1,335.84,2026-10-18T18:42:31.514Z
5094,Lesley Loeb,F,Raw,null,Amateur Master 55-59,65.95,67.5,null,null,null,58.97,null,122.47,181.44,1,188.33,2026-10-18T18:42:31.514Z
5094,Pamela Dragosh,F,Raw,null,Amateur Master 45-49,91.08,100,null,null,null,61.23,null,131.54,192.78,1,165.77,2026-10-18T18:42:31.514Z
5094,Louise Chenal,F,Raw,null,Amateur Master 55-59,89.45,90,null,null,null,54.43,null,129.27,183.7,1,159.14,2026-10-18T18:42:31.514Z


In [0]:
%sql
-- Silver layer: cleaned, deduplicated, typed tables.
-- Bronze was appended (full Kaggle re-load each run), so dedupe keeps the latest
-- ingested batch per natural key and drops load-provenance columns.

CREATE SCHEMA IF NOT EXISTS workspace.powerlifting_silver;

-- meets: natural key = MeetID (unique within a snapshot). Keep latest batch.
CREATE OR REPLACE TABLE workspace.powerlifting_silver.meets AS
WITH latest AS (
  SELECT *,
    ROW_NUMBER() OVER (PARTITION BY MeetID ORDER BY _ingested_at DESC) AS _rn
  FROM workspace.powerlifting_bronze.meets
)
SELECT
  MeetID,
  trim(MeetPath)    AS MeetPath,
  trim(Federation)  AS Federation,
  to_date(Date, 'yyyy-MM-dd') AS MeetDate,
  trim(MeetCountry) AS MeetCountry,
  trim(MeetState)   AS MeetState,
  trim(MeetTown)    AS MeetTown,
  trim(MeetName)    AS MeetName
FROM latest
WHERE _rn = 1;

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- openpowerlifting: natural key = (MeetID, Name, Sex, Equipment, Division, WeightClassKg).
-- A lifter can appear multiple times under the same key with partial results
-- (e.g. bench-only vs full-power) or exact source duplicates. Dedupe keeps the
-- latest ingested batch, then the row with the highest TotalKg (most complete result).
CREATE OR REPLACE TABLE workspace.powerlifting_silver.openpowerlifting AS
WITH latest AS (
  SELECT *,
    ROW_NUMBER() OVER (
      PARTITION BY MeetID, Name, Sex, Equipment, Division, WeightClassKg
      ORDER BY _ingested_at DESC, TotalKg DESC NULLS LAST, BestSquatKg DESC NULLS LAST
    ) AS _rn
  FROM workspace.powerlifting_bronze.openpowerlifting
)
SELECT
  MeetID,
  trim(Name)        AS Name,
  Sex,
  CASE Sex WHEN 'M' THEN 'Male' WHEN 'F' THEN 'Female' ELSE NULL END AS SexLabel,
  trim(Equipment)   AS Equipment,
  CAST(Age AS DECIMAL(5,1)) AS Age,
  trim(Division)    AS Division,
  BodyweightKg,
  trim(WeightClassKg) AS WeightClassKg,
  -- 4th attempts: negative = failed attempt -> NULL
  CASE WHEN Squat4Kg    >= 0 THEN Squat4Kg    END AS Squat4Kg,
  CASE WHEN Bench4Kg     >= 0 THEN Bench4Kg     END AS Bench4Kg,
  CASE WHEN Deadlift4Kg  >= 0 THEN Deadlift4Kg  END AS Deadlift4Kg,
  -- Best lifts: negative = no successful lift -> NULL
  CASE WHEN BestSquatKg    >= 0 THEN BestSquatKg    END AS BestSquatKg,
  CASE WHEN BestBenchKg     >= 0 THEN BestBenchKg     END AS BestBenchKg,
  CASE WHEN BestDeadliftKg  >= 0 THEN BestDeadliftKg  END AS BestDeadliftKg,
  TotalKg,
  -- Place: numeric placement -> INT, non-numeric codes -> NULL
  CASE WHEN Place RLIKE '^[0-9]+$' THEN CAST(Place AS INT) END AS Place,
  -- FinishStatus enum decoded from Place codes
  CASE
    WHEN Place RLIKE '^[0-9]+$' THEN 'Finished'
    WHEN Place = 'DQ' THEN 'Disqualified'
    WHEN Place = 'G'  THEN 'Guest'
    WHEN Place = 'DD' THEN 'Did Not Weigh In'
    WHEN Place = 'NS' THEN 'No Show'
    ELSE NULL
  END AS FinishStatus,
  Wilks
FROM latest
WHERE _rn = 1;

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Verify silver layer
SELECT 'meets' AS table_name, COUNT(*) AS row_count FROM workspace.powerlifting_silver.meets
UNION ALL
SELECT 'openpowerlifting', COUNT(*) FROM workspace.powerlifting_silver.openpowerlifting;

-- FinishStatus distribution (should account for all non-null Place)
SELECT FinishStatus, COUNT(*) AS cnt
FROM workspace.powerlifting_silver.openpowerlifting
GROUP BY 1
ORDER BY 2 DESC;

-- Confirm no negative values survived in lift columns
SELECT
  SUM(CASE WHEN BestSquatKg    < 0 THEN 1 ELSE 0 END) AS neg_squat,
  SUM(CASE WHEN BestBenchKg     < 0 THEN 1 ELSE 0 END) AS neg_bench,
  SUM(CASE WHEN BestDeadliftKg  < 0 THEN 1 ELSE 0 END) AS neg_dead,
  SUM(CASE WHEN Squat4Kg    < 0 THEN 1 ELSE 0 END) AS neg_squat4,
  SUM(CASE WHEN Place IS NOT NULL AND FinishStatus IS NULL THEN 1 ELSE 0 END) AS place_without_status,
  SUM(CASE WHEN Place IS NULL AND FinishStatus IS NOT NULL THEN 1 ELSE 0 END) AS status_without_place
FROM workspace.powerlifting_silver.openpowerlifting;

-- Sample cleaned rows
SELECT * FROM workspace.powerlifting_silver.openpowerlifting LIMIT 5;

MeetID,Name,Sex,SexLabel,Equipment,Age,Division,BodyweightKg,WeightClassKg,Squat4Kg,Bench4Kg,Deadlift4Kg,BestSquatKg,BestBenchKg,BestDeadliftKg,TotalKg,Place,FinishStatus,Wilks
0,Al Reiss,M,Male,Wraps,61.0,Mst 60-64,101.79,110,176.9,null,null,170.1,104.33,167.83,442.26,1,Finished,267.3
0,Alexis Eliopoulos,F,Female,Wraps,30.0,Senior,88.36,90,null,null,204.12,174.63,122.47,195.04,492.14,1,Finished,428.59
0,Andrew Mooe,M,Male,Wraps,27.0,Open Senior,99.25,100,null,null,null,226.8,163.29,272.16,662.25,2,Finished,404.26
0,Angie Belk Terry,F,Female,Wraps,47.0,Mst 45-49,59.6,60,null,null,null,47.63,20.41,70.31,138.35,1,Finished,155.05
0,Barron Henderson,M,Male,Single-ply,40.0,Mst 40-44,118.66,125,null,null,null,362.87,181.44,301.64,845.95,1,Finished,487.65


In [0]:
%sql
-- Gold layer: business-ready analytical tables for reporting and BI.
-- Joins silver.openpowerlifting with silver.meets, adds derived dimensions,
-- and pre-computes common aggregations.

CREATE SCHEMA IF NOT EXISTS workspace.powerlifting_gold;

-- dim_meets: enriched meet dimension with temporal and geographic attributes.
-- ParticipantCount is precomputed for fast meet-level filtering.
CREATE OR REPLACE TABLE workspace.powerlifting_gold.dim_meets AS
SELECT
  m.MeetID,
  m.MeetPath,
  m.Federation,
  m.MeetDate,
  YEAR(m.MeetDate)         AS MeetYear,
  FLOOR(YEAR(m.MeetDate) / 10) * 10 AS MeetDecade,
  m.MeetCountry,
  m.MeetState,
  m.MeetTown,
  m.MeetName,
  COUNT(op.MeetID)         AS ParticipantCount
FROM workspace.powerlifting_silver.meets m
LEFT JOIN workspace.powerlifting_silver.openpowerlifting op ON m.MeetID = op.MeetID
GROUP BY m.MeetID, m.MeetPath, m.Federation, m.MeetDate,
         m.MeetCountry, m.MeetState, m.MeetTown, m.MeetName;

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- fact_meet_results: enriched one-row-per-lifter-per-meet fact table.
-- Denormalises meet info for convenience, adds EventCategory, AgeCategory,
-- and StrengthRatio (total / bodyweight) for direct analytical use.
CREATE OR REPLACE TABLE workspace.powerlifting_gold.fact_meet_results AS
SELECT
  op.MeetID,
  op.Name,
  op.Sex,
  op.SexLabel,
  op.Equipment,
  op.Age,
  op.Division,
  op.BodyweightKg,
  op.WeightClassKg,
  op.BestSquatKg,
  op.BestBenchKg,
  op.BestDeadliftKg,
  op.Squat4Kg,
  op.Bench4Kg,
  op.Deadlift4Kg,
  op.TotalKg,
  op.Place,
  op.FinishStatus,
  op.Wilks,
  -- Denormalised meet columns
  m.MeetDate,
  m.MeetName,
  m.Federation,
  m.MeetCountry,
  m.MeetState,
  m.MeetTown,
  YEAR(m.MeetDate)         AS MeetYear,
  FLOOR(YEAR(m.MeetDate) / 10) * 10 AS MeetDecade,
  -- Event category derived from which lifts were attempted
  CASE
    WHEN op.BestSquatKg IS NOT NULL AND op.BestBenchKg IS NOT NULL AND op.BestDeadliftKg IS NOT NULL THEN 'Full Power'
    WHEN op.BestBenchKg IS NOT NULL AND op.BestSquatKg IS NULL AND op.BestDeadliftKg IS NULL THEN 'Bench Only'
    WHEN op.BestSquatKg IS NOT NULL AND op.BestBenchKg IS NULL AND op.BestDeadliftKg IS NOT NULL THEN 'Squat-Deadlift'
    WHEN op.BestSquatKg IS NOT NULL AND op.BestBenchKg IS NULL AND op.BestDeadliftKg IS NULL THEN 'Squat Only'
    WHEN op.BestDeadliftKg IS NOT NULL AND op.BestSquatKg IS NULL AND op.BestBenchKg IS NULL THEN 'Deadlift Only'
    ELSE 'Other'
  END AS EventCategory,
  -- Strength ratio: total lifted relative to bodyweight
  CASE WHEN op.BodyweightKg > 0 AND op.TotalKg IS NOT NULL
       THEN ROUND(op.TotalKg / op.BodyweightKg, 3)
  END AS StrengthRatio,
  -- Age category (IPF-style bands)
  CASE
    WHEN op.Age < 18  THEN 'Youth'
    WHEN op.Age BETWEEN 18 AND 23  THEN 'Junior'
    WHEN op.Age BETWEEN 24 AND 34  THEN 'Open'
    WHEN op.Age BETWEEN 35 AND 49  THEN 'Sub-Master'
    WHEN op.Age >= 50 THEN 'Master'
    ELSE 'Unknown'
  END AS AgeCategory,
  CASE WHEN op.FinishStatus = 'Finished' THEN TRUE ELSE FALSE END AS IsCompleted
FROM workspace.powerlifting_silver.openpowerlifting op
LEFT JOIN workspace.powerlifting_silver.meets m ON op.MeetID = m.MeetID;

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- agg_meet_summary: one row per meet with participant counts, lift stats,
-- sex breakdown, equipment distribution, and the top lifter by TotalKg.
CREATE OR REPLACE TABLE workspace.powerlifting_gold.agg_meet_summary AS
SELECT
  m.MeetID,
  m.MeetName,
  m.MeetDate,
  YEAR(m.MeetDate) AS MeetYear,
  m.Federation,
  m.MeetCountry,
  COUNT(*)                                       AS TotalParticipants,
  SUM(CASE WHEN op.Sex = 'M' THEN 1 ELSE 0 END)  AS MaleParticipants,
  SUM(CASE WHEN op.Sex = 'F' THEN 1 ELSE 0 END)  AS FemaleParticipants,
  ROUND(AVG(op.TotalKg), 2)                       AS AvgTotalKg,
  MAX(op.TotalKg)                                 AS MaxTotalKg,
  ROUND(AVG(op.Wilks), 2)                          AS AvgWilks,
  MAX(op.Wilks)                                   AS MaxWilks,
  SUM(CASE WHEN op.FinishStatus = 'Finished' THEN 1 ELSE 0 END)       AS FinishedCount,
  SUM(CASE WHEN op.FinishStatus = 'Disqualified' THEN 1 ELSE 0 END)  AS DisqualifiedCount,
  MAX_BY(op.Name, op.TotalKg)                     AS TopLifterName,
  SUM(CASE WHEN op.Equipment = 'Raw'       THEN 1 ELSE 0 END) AS RawCount,
  SUM(CASE WHEN op.Equipment = 'Single-ply' THEN 1 ELSE 0 END) AS SinglePlyCount,
  SUM(CASE WHEN op.Equipment = 'Multi-ply' THEN 1 ELSE 0 END) AS MultiPlyCount,
  SUM(CASE WHEN op.Equipment = 'Wraps'      THEN 1 ELSE 0 END) AS WrapsCount
FROM workspace.powerlifting_silver.openpowerlifting op
JOIN workspace.powerlifting_silver.meets m ON op.MeetID = m.MeetID
GROUP BY m.MeetID, m.MeetName, m.MeetDate, YEAR(m.MeetDate), m.Federation, m.MeetCountry;

-- agg_lifter_summary: one row per lifter with career stats, personal records,
-- federation diversity, and favourite equipment type.
CREATE OR REPLACE TABLE workspace.powerlifting_gold.agg_lifter_summary AS
WITH lifter_agg AS (
  SELECT
    op.Name,
    op.Sex,
    op.SexLabel,
    COUNT(DISTINCT op.MeetID)        AS TotalMeets,
    MIN(m.MeetDate)                  AS FirstMeetDate,
    MAX(m.MeetDate)                  AS LastMeetDate,
    DATEDIFF(MAX(m.MeetDate), MIN(m.MeetDate)) AS CareerSpanDays,
    MAX(op.TotalKg)                  AS BestTotalKg,
    MAX(op.BestSquatKg)              AS BestSquatKg,
    MAX(op.BestBenchKg)              AS BestBenchKg,
    MAX(op.BestDeadliftKg)           AS BestDeadliftKg,
    ROUND(AVG(op.Wilks), 2)          AS AvgWilks,
    MAX(op.Wilks)                    AS BestWilks,
    COUNT(DISTINCT m.Federation)     AS DistinctFederations
  FROM workspace.powerlifting_silver.openpowerlifting op
  JOIN workspace.powerlifting_silver.meets m ON op.MeetID = m.MeetID
  GROUP BY op.Name, op.Sex, op.SexLabel
),
equipment_ranked AS (
  SELECT
    Name, Sex, Equipment,
    ROW_NUMBER() OVER (PARTITION BY Name, Sex ORDER BY COUNT(*) DESC) AS eq_rn
  FROM workspace.powerlifting_silver.openpowerlifting
  GROUP BY Name, Sex, Equipment
)
SELECT
  la.*,
  er.Equipment AS FavouriteEquipment
FROM lifter_agg la
LEFT JOIN equipment_ranked er ON la.Name = er.Name AND la.Sex = er.Sex AND er.eq_rn = 1;

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Verify gold layer: row counts and spot-checks
SELECT 'dim_meets' AS table_name, COUNT(*) AS row_count FROM workspace.powerlifting_gold.dim_meets
UNION ALL
SELECT 'fact_meet_results', COUNT(*) FROM workspace.powerlifting_gold.fact_meet_results
UNION ALL
SELECT 'agg_meet_summary', COUNT(*) FROM workspace.powerlifting_gold.agg_meet_summary
UNION ALL
SELECT 'agg_lifter_summary', COUNT(*) FROM workspace.powerlifting_gold.agg_lifter_summary;

-- EventCategory distribution
SELECT EventCategory, COUNT(*) AS cnt
FROM workspace.powerlifting_gold.fact_meet_results
GROUP BY 1
ORDER BY 2 DESC;

-- AgeCategory distribution
SELECT AgeCategory, COUNT(*) AS cnt
FROM workspace.powerlifting_gold.fact_meet_results
GROUP BY 1
ORDER BY 2 DESC;

-- Confirm fact table join coverage (no unmatched meets)
SELECT
  SUM(CASE WHEN MeetDate IS NULL THEN 1 ELSE 0 END) AS unmatched_meets,
  SUM(CASE WHEN MeetDate IS NOT NULL THEN 1 ELSE 0 END) AS matched_meets
FROM workspace.powerlifting_gold.fact_meet_results;

-- Top 10 lifters by best TotalKg
SELECT Name, SexLabel, BestTotalKg, BestSquatKg, BestBenchKg, BestDeadliftKg,
       BestWilks, TotalMeets, FavouriteEquipment
FROM workspace.powerlifting_gold.agg_lifter_summary
ORDER BY BestTotalKg DESC NULLS LAST
LIMIT 10;

Name,SexLabel,BestTotalKg,BestSquatKg,BestBenchKg,BestDeadliftKg,BestWilks,TotalMeets,FavouriteEquipment
Dave Hoff,Male,1365.31,560.19,455.86,374.21,779.38,23,Multi-ply
Donnie Thompson,Male,1360.78,573.79,430.91,356.07,735.88,4,Multi-ply
AJ Roberts,Male,1295.01,546.58,412.77,369.68,725.48,12,Multi-ply
Blaine Sumner,Male,1272.5,505.0,410.0,370.0,692.21,20,Single-ply
Shane Hammock,Male,1270.06,508.02,396.89,399.16,714.43,11,Multi-ply
Jake Anderson,Male,1270.06,521.63,376.48,396.89,707.79,14,Multi-ply
Jeff Frank,Male,1257.5,552.5,367.5,340.0,675.9,15,Multi-ply
Josh Conley,Male,1249.65,489.88,374.21,408.23,686.71,3,Multi-ply
Brian Carroll,Male,1238.31,537.51,374.21,353.8,707.94,13,Multi-ply
Chris Janek,Male,1236.04,501.22,356.07,381.02,690.92,9,Multi-ply
